# Module 5: Production Deployment & MLOps
## Task 5: The "Boxed" AI (Dockerization)

### **Goal**
Make your system portable and resilient. 
In this notebook, we will:
1.  **Write a Dockerfile** for our FastAPI + Mistral backend.
2.  **Create a Docker Compose** file to link the API and the Vector DB.
3.  **Implement Logging** to track agent "Traces" during execution.

### **Step 1: The Dockerfile**
The Dockerfile is the "Recipe" for your environment. We use a lightweight Python image and install only what is necessary to run our Mistral Agent.

In [3]:
dockerfile_content = """
# Use an optimized Python 3.14.3 image
FROM python:3.14.3-slim

# Set the working directory
WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y \\
    build-essential \\
    && rm -rf /var/lib/apt/lists/*

# Copy requirements and install
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy the application code
COPY . .

# Expose the FastAPI port
EXPOSE 8000

# Command to run the production server
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile_content)

print("--- Dockerfile successfully created ---")

--- Dockerfile successfully created ---


### **Step 2: Docker Compose**
Instead of starting five different terminal windows, we use **Docker Compose**. This file defines how our API and our Vector Database (Qdrant) live together.

In [4]:
compose_content = """
version: '3.8'

services:
  # The Mistral Agent API we built in Module 3
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - MISTRAL_API_KEY=${MISTRAL_API_KEY}
    depends_on:
      - vector_db

  # The Vector Database for the Memory Vault
  vector_db:
    image: qdrant/qdrant
    ports:
      - "6333:6333"
"""

with open("docker-compose.yml", "w") as f:
    f.write(compose_content)

print("--- docker-compose.yml successfully created ---")

--- docker-compose.yml successfully created ---


### **Step 3: Tracking Agent Traces**
In production, we need to know what the Agent did. We implement a logging wrapper to capture every "Thought" and "Action" the agent takes.

In [5]:
import logging

# Configure Industrial Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - [AGENT-TRACE] - %(message)s'
)

def log_agent_step(step_name, data):
    """Captures a specific trace in the agentic loop"""
    logging.info(f"Step: {step_name} | Metadata: {data}")

# Example Usage during an agent loop
log_agent_step("Tool_Selection", {"tool": "get_weather", "input": "Peshawar"})

2026-05-08 11:37:43,354 - [AGENT-TRACE] - Step: Tool_Selection | Metadata: {'tool': 'get_weather', 'input': 'Peshawar'}


### **Final Step: Deployment**
To launch your boxed system, ensure Docker is running on your machine, then run:

```bash
docker-compose up --build